# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)
**Document:** EU Artificial Intelligence Act (Regulation (EU) 2024/1689)

Two pipelines over the *same* chunked corpus:
1. **Baseline:** Bi-Encoder -> Top 10 (cosine) -> Top 3 straight to the LLM
2. **Re-ranking:** same Top 10 -> Cross-Encoder re-scores all 10 -> Top 3 re-ranked to the LLM

Everything you may want to change (LLM, models, PDF path, chunk size) is in the **config cell** below.

## 1. Setup

In [ ]:
! pip install -q sentence-transformers faiss-cpu pypdf pandas google-genai openai anthropic

## 2. Config (change the LLM here and nowhere else)

| Provider | Example `LLM_MODEL` | Key env var |
|---|---|---|
| `gemini` | `gemini-2.5-flash` | `GEMINI_API_KEY` |
| `openai` | `gpt-4o-mini` | `OPENAI_API_KEY` |
| `anthropic` | any current Claude model id | `ANTHROPIC_API_KEY` |
| `ollama` | `llama3.1:8b` | none (local server) |

In Colab, add the key under the key icon (Secrets) with the same name as the env var.

In [35]:

import os
import time
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load your existing .env
load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        "GEMINI_API_KEY was not loaded. "
        "Check the .env path and variable name."
    )

client = genai.Client(api_key=api_key)

# One tiny request per listed text-generation model
TEST_PROMPT = "Reply with exactly: OK"
MAX_OUTPUT_TOKENS = 5

results = []

print("Fetching models available to your API key...")

try:
    models = list(client.models.list())
except Exception as e:
    print("Could not list models:", str(e))
    raise

candidates = []

for model in models:
    name = getattr(model, "name", None)
    actions = getattr(model, "supported_actions", []) or []

    if not name:
        continue

    # Only test models advertising content generation.
    if any("generateContent" in str(a) for a in actions):
        candidates.append(name.replace("models/", ""))

print(f"Found {len(candidates)} candidate models.")
print("Testing them one at a time...\n")

for model_name in candidates:
    start = time.perf_counter()

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=TEST_PROMPT,
            config=types.GenerateContentConfig(
                max_output_tokens=MAX_OUTPUT_TOKENS,
                temperature=0
            )
        )

        elapsed = time.perf_counter() - start
        answer = (response.text or "").strip()

        if answer:
            results.append({
                "model": model_name,
                "status": "WORKS",
                "seconds": round(elapsed, 2),
                "sample": answer[:80],
                "error": ""
            })
            print(f"OK      {model_name:<40} {elapsed:.2f}s")
        else:
            print(f"NO TEXT {model_name}")
            results.append({
                "model": model_name,
                "status": "NO TEXT",
                "seconds": round(elapsed, 2),
                "sample": "",
                "error": "Empty response"
            })

    except Exception as e:
        message = str(e).replace("\n", " ")[:180]
        print(f"FAILED  {model_name:<40} {message}")
        results.append({
            "model": model_name,
            "status": "FAILED",
            "seconds": round(time.perf_counter() - start, 2),
            "sample": "",
            "error": message
        })

    # Avoid hammering the API and triggering rate limits.
    time.sleep(1)

print("\n" + "=" * 70)

working = [r for r in results if r["status"] == "WORKS"]
working.sort(key=lambda r: r["seconds"])

if working:
    print("WINNER: FASTEST MODEL THAT RESPONDED")
    print("Model:", working[0]["model"])
    print("Latency:", working[0]["seconds"], "seconds")

    print("\nOther working models:")
    for r in working:
        print(f'{r["model"]:<40} {r["seconds"]:.2f}s')

    # Save the selected model for your existing configuration.
    WINNING_MODEL = working[0]["model"]
    print("\nSet LLM_PROVIDER = 'gemini'")
    print("Set LLM_MODEL =", repr(WINNING_MODEL))
else:
    print("No model succeeded.")
    print("Inspect the FAILED messages above.")

# Show complete results in a table
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)


Fetching models available to your API key...
Found 46 candidate models.
Testing them one at a time...

FAILED  gemini-2.5-flash                         404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash fo
FAILED  gemini-2.5-pro                           404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use models/gemini-3.1-pro-previe
FAILED  gemini-2.5-flash-preview-tts             400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The requested combination of response modalities (TEXT) is not supported by the model. models/gemini-2.5-flash-preview-tts
FAILED  gemini-2.5-pro-preview-tts               429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information 

KeyboardInterrupt: 

In [36]:
# ---------------- LLM (swap freely) ----------------
LLM_PROVIDER = "gemini"            # "gemini" | "openai" | "anthropic" | "ollama"
LLM_MODEL    = "gemini-3.1-flash-lite-preview"  # model name for the chosen provider
LLM_TEMPERATURE = 0.0              # keep 0 so Answer 1 vs Answer 2 differ only because of the context

# ---------------- Retrieval models ----------------
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # Bi-Encoder
CROSS_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # Cross-Encoder

# ---------------- Document + chunking ----------------
PDF_PATH = "doc.pdf"   # upload the PDF next to the notebook
SKIP_RECITALS = True         # start from the operative text ("HAVE ADOPTED THIS REGULATION")
CHUNK_WORDS   = 250          # max words per chunk
CHUNK_OVERLAP = 40           # words of overlap between consecutive chunks of the same section

# ---------------- Retrieval ----------------
TOP_K_RETRIEVE = 10          # candidates from the Bi-Encoder
TOP_K_CONTEXT  = 3           # chunks given to the LLM

In [ ]:
import os, re, time, textwrap
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---- API key handling (Colab secrets first, then environment) ----
KEY_ENV = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY",
           "anthropic": "ANTHROPIC_API_KEY", "ollama": None}

from dotenv import load_dotenv
import os

load_dotenv()

def get_api_key():
    name = KEY_ENV[LLM_PROVIDER]
    if name is None:
        return None
    return os.environ.get(name)



def call_llm(prompt: str) -> str:
    """Single entry point for the LLM. Both pipelines use this, so changing
    LLM_PROVIDER / LLM_MODEL in the config cell is all it takes to switch models."""
    key = get_api_key()
    if LLM_PROVIDER == "gemini":
        from google import genai
        client = genai.Client(api_key=key)
        r = client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config={"temperature": LLM_TEMPERATURE})
        return r.text
    if LLM_PROVIDER == "openai":
        from openai import OpenAI
        r = OpenAI(api_key=key).chat.completions.create(
            model=LLM_MODEL, temperature=LLM_TEMPERATURE,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if LLM_PROVIDER == "anthropic":
        import anthropic
        r = anthropic.Anthropic(api_key=key).messages.create(
            model=LLM_MODEL, max_tokens=800,
            messages=[{"role": "user", "content": prompt}])
        return r.content[0].text
    if LLM_PROVIDER == "ollama":
        import requests
        r = requests.post("http://localhost:11434/api/generate",
                          json={"model": LLM_MODEL, "prompt": prompt, "stream": False,
                                "options": {"temperature": LLM_TEMPERATURE}})
        return r.json()["response"]
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

## 3. Load the PDF and build chunks
Strategy: split on legal structure (`Article N` / `ANNEX X` headings) so a chunk never mixes two articles,
then cap long articles at `CHUNK_WORDS` words with a small overlap. Each chunk keeps its article label and page,
which makes failure analysis easy to read. Both pipelines use exactly this same corpus.

In [5]:
reader = PdfReader(PDF_PATH)
print("Pages:", len(reader.pages))

# Remove running headers/footers of the Official Journal layout
NOISE = re.compile(r"^(OJ L, \d{1,2}\.\d{1,2}\.\d{4}|EN|ELI: .*|\d+/\d+)\s*$")

pages = []
for i, p in enumerate(reader.pages, start=1):
    lines = [ln for ln in (p.extract_text() or "").splitlines() if not NOISE.match(ln.strip())]
    pages.append((i, "\n".join(lines)))

full = "\n".join(f"<<P{i}>>\n{t}" for i, t in pages)

if SKIP_RECITALS:
    m = re.search(r"HAVE ADOPTED THIS REGULATION", full)
    if m:
        full = full[m.end():]
    else:
        print("Marker not found; keeping the whole document.")

print("Characters:", len(full))

Pages: 144
Characters: 351142


In [6]:
PAGE_MARK = re.compile(r"<<P(\d+)>>")
HEADING = re.compile(r"(?m)^((?:Article \d+)|(?:ANNEX [IVXL]+))\s*$")

parts = HEADING.split(full)   # [pre, heading, body, heading, body, ...]
chunks = []                   # list of dicts: id, section, page, text
cur_page = 1

def add_section(label, body, start_page):
    words = PAGE_MARK.sub(" ", body).split()
    if not words:
        return
    # first line of the body is normally the article title
    title = body.strip().splitlines()[0].strip() if body.strip() else ""
    title = PAGE_MARK.sub("", title).strip()
    header = f"[{label} - {title}]" if title else f"[{label}]"
    step = CHUNK_WORDS - CHUNK_OVERLAP
    for s in range(0, len(words), step):
        piece = words[s:s + CHUNK_WORDS]
        if len(piece) < 30 and s > 0:   # drop tiny tail fragments
            break
        chunks.append({"id": len(chunks), "section": label, "page": start_page,
                       "text": header + " " + " ".join(piece)})
        if s + CHUNK_WORDS >= len(words):
            break

# text before the first heading (e.g. Chapter I intro) is skipped on purpose
for pm in PAGE_MARK.finditer(parts[0]):
    cur_page = int(pm.group(1))

for j in range(1, len(parts), 2):
    label, body = parts[j].strip(), parts[j + 1]
    add_section(label, body, cur_page)
    for pm in PAGE_MARK.finditer(body):
        cur_page = int(pm.group(1))

print("Total chunks:", len(chunks))
print("Avg words/chunk:", int(np.mean([len(c['text'].split()) for c in chunks])))
print("\nSample chunk:\n", textwrap.fill(chunks[5]["text"][:600], 110))

Total chunks: 300
Avg words/chunk: 211

Sample chunk:
 [Article 3 - Definitions] Definitions For the purposes of this Regulation, the following definitions apply:
(1) ‘AI system’ means a machine-based system that is designed to operate with varying levels of autonomy and
that may exhibit adaptiveness after deployment, and that, for explicit or implicit objectives, infers, from
the input it receives, how to generate outputs such as predictions, content, recommendations, or decisions
that can influence physical or virtual environments; (2) ‘risk’ means the combination of the probability of an
occurrence of harm and the severity of that harm; (3) ‘pr


## 4. Index with the Bi-Encoder (FAISS, inner product on normalised vectors = cosine)

In [7]:
bi_encoder = SentenceTransformer(EMBED_MODEL)
t0 = time.perf_counter()
emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=64,
                        normalize_embeddings=True, show_progress_bar=True).astype("float32")
print(f"Indexed {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s, dim={emb.shape[1]}")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

cross_encoder = CrossEncoder(CROSS_MODEL)

Batches: 100%|██████████| 5/5 [00:01<00:00,  4.56it/s]


Indexed 300 chunks in 1.1s, dim=384


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1675.91it/s]


## 5. The two pipelines
Retrieval and re-ranking are timed separately from the LLM call so the latency table is clean.

In [8]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Stage 1: Bi-Encoder -> top-k candidates (id, cosine score), best first."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates):
    """Stage 2: Cross-Encoder scores every (query, chunk) pair jointly, then re-sorts."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = [(cid, float(s)) for (cid, _), s in zip(candidates, ce_scores)]
    return sorted(out, key=lambda x: x[1], reverse=True)

PROMPT_TEMPLATE = """You are answering questions about the EU Artificial Intelligence Act.
Use ONLY the context below. If the context does not contain the answer, say so.
Mention any exceptions or conditions that appear in the context.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(f"[Chunk {n+1}] {chunks[cid]['text']}" for n, cid in enumerate(chunk_ids))
    return PROMPT_TEMPLATE.format(context=ctx, question=query)   # identical for both pipelines

def run_pipelines(query, with_llm=True):
    res = {"query": query}
    t = time.perf_counter()
    cands = baseline_retrieve(query)
    res["t_retrieve"] = time.perf_counter() - t

    t = time.perf_counter()
    reranked = rerank(query, cands)
    res["t_rerank"] = time.perf_counter() - t

    res["baseline_rank"] = [cid for cid, _ in cands]           # ranks 1..10
    res["rerank_rank"]   = [cid for cid, _ in reranked]
    res["baseline_top3"] = res["baseline_rank"][:TOP_K_CONTEXT]
    res["rerank_top3"]   = res["rerank_rank"][:TOP_K_CONTEXT]
    res["bi_scores"] = dict(cands)
    res["ce_scores"] = dict(reranked)

    if with_llm:
        t = time.perf_counter(); res["answer1"] = call_llm(build_prompt(query, res["baseline_top3"])); res["t_llm1"] = time.perf_counter() - t
        t = time.perf_counter(); res["answer2"] = call_llm(build_prompt(query, res["rerank_top3"]));   res["t_llm2"] = time.perf_counter() - t
    return res

## 6. Failure-mode stress testing
For each query we define a **gold pattern**: a regex that identifies the chunk(s) that actually contain the
answer (you verify this by reading the Act). The helper reports the gold chunk's rank before and after re-ranking.

**Important:** the queries below are *starting candidates*. Read the Act, confirm each gold pattern, and
replace/extend them with failures you actually observe. The scanner in the next cell helps you find them.

In [9]:
def gold_rank(rank_list, gold_pattern):
    """1-based rank of the best-placed chunk matching gold_pattern, else None."""
    rx = re.compile(gold_pattern, re.I | re.S)
    for pos, cid in enumerate(rank_list, start=1):
        if rx.search(chunks[cid]["text"]):
            return pos
    return None

# category, query, gold_pattern (VERIFY each against the PDF)
TEST_QUERIES = [
    ("Exceptions",   "Is emotion recognition at work banned, and are there any exceptions?",
                     r"emotions? of a natural person.{0,300}(medical|safety)"),
    ("Exceptions",   "When can police use real-time remote biometric identification in public spaces?",
                     r"real-time.{0,200}remote biometric identification.{0,600}(search for|prevention of|localisation)"),
    ("Negation",     "Which AI practices are not allowed in the EU?",
                     r"(shall be prohibited|following AI practices)"),
    ("Negation",     "Can AI systems scrape facial images from CCTV to build recognition databases?",
                     r"untargeted scraping of facial images"),
    ("Keyword trap", "Who is liable if a deployer modifies a high-risk system?",
                     r"(considered to be a provider|substantial modification)"),
    ("Document-specific", "What fines apply for violating the prohibited practices rules?",
                     r"(35 000 000|7 ?%)"),
]

def analyse(category, query, gold_pattern, with_llm=False):
    r = run_pipelines(query, with_llm=with_llm)
    b = gold_rank(r["baseline_rank"], gold_pattern)
    c = gold_rank(r["rerank_rank"],  gold_pattern)
    r.update(category=category, gold_pattern=gold_pattern, gold_baseline_rank=b, gold_rerank_rank=c)
    return r

scan = [analyse(cat, q, g, with_llm=False) for cat, q, g in TEST_QUERIES]
summary = pd.DataFrame([{
    "category": r["category"], "query": r["query"][:60],
    "gold rank (baseline)": r["gold_baseline_rank"] or "not in top10",
    "gold rank (re-rank)":  r["gold_rerank_rank"]  or "not in top10",
    "in top3 baseline": (r["gold_baseline_rank"] or 99) <= TOP_K_CONTEXT,
    "in top3 re-rank":  (r["gold_rerank_rank"]  or 99) <= TOP_K_CONTEXT,
} for r in scan])
summary

,category,query,gold rank (baseline),gold rank (re-rank),in top3 baseline,in top3 re-rank
0,Exceptions,"Is emotion recognition at work banned, and are...",2,2,True,True
1,Exceptions,When can police use real-time remote biometric...,5,9,False,False
2,Negation,Which AI practices are not allowed in the EU?,not in top10,not in top10,False,False
3,Negation,Can AI systems scrape facial images from CCTV ...,1,1,True,True
4,Keyword trap,Who is liable if a deployer modifies a high-ri...,7,1,False,True
5,Document-specific,What fines apply for violating the prohibited ...,2,1,True,True


**How to use this:** the good "failure queries" for the report are rows where the gold chunk is **outside the
baseline top 3 but inside the re-ranked top 3**. If a query does not show that, tweak the wording or pick a
different clause. Rows where re-ranking does *not* help are also worth a sentence in your analysis.

### 6a. Inspect one query in detail (rank shifts + both answers)

In [33]:
print("Provider:", LLM_PROVIDER)
print("Expected key:", KEY_ENV.get(LLM_PROVIDER))
print("Working directory:", os.getcwd())
print("Key loaded:", bool(get_api_key()))

Provider: openai
Expected key: OPENAI_API_KEY
Working directory: c:\Users\DEDEEPYA\Downloads
Key loaded: True


In [37]:
def show(r, n=10):
    print("QUERY:", r["query"], f"[{r.get('category','')}]\n")
    print(f"{'BaseRank':<9}{'ChunkID':<9}{'Section':<12}{'BiScore':<9}{'CE Score':<10}{'ReRank':<7}")
    rr = {cid: pos for pos, cid in enumerate(r["rerank_rank"], start=1)}
    for pos, cid in enumerate(r["baseline_rank"][:n], start=1):
        print(f"{pos:<9}{cid:<9}{chunks[cid]['section']:<12}{r['bi_scores'][cid]:<9.3f}{r['ce_scores'][cid]:<10.3f}{rr[cid]:<7}")
    if "answer1" in r:
        print("\n--- Answer 1 (baseline top 3) ---\n", r["answer1"])
        print("\n--- Answer 2 (re-ranked top 3) ---\n", r["answer2"])

# pick a row index from the summary table above
i = 0
full_run = analyse(*TEST_QUERIES[i], with_llm=True)
print("Gold chunk rank:", full_run["gold_baseline_rank"], "->", full_run["gold_rerank_rank"], "\n")
show(full_run)

Gold chunk rank: 2 -> 2 

QUERY: Is emotion recognition at work banned, and are there any exceptions? [Exceptions]

BaseRank ChunkID  Section     BiScore  CE Score  ReRank 
1        122      Article 50  0.430    -0.907    1      
2        21       Article 5   0.424    -1.951    2      
3        267      ANNEX III   0.317    -7.000    4      
4        19       Article 5   0.308    -8.372    5      
5        11       Article 3   0.307    -3.200    3      
6        27       Article 5   0.292    -10.340   6      
7        4        Article 2   0.285    -11.075   9      
8        3        Article 2   0.271    -11.221   10     
9        26       Article 5   0.269    -10.888   8      
10       25       Article 5   0.258    -10.559   7      

--- Answer 1 (baseline top 3) ---
 Yes, the use of AI systems to infer emotions of a natural person in the workplace is prohibited under Article 5.

There is one exception: the prohibition does not apply if the AI system is intended to be put in place or i

### 6b. Run every test query end-to-end (answers for the report)

In [38]:
all_runs = [analyse(cat, q, g, with_llm=True) for cat, q, g in TEST_QUERIES]
for r in all_runs:
    print("=" * 100)
    show(r)

QUERY: Is emotion recognition at work banned, and are there any exceptions? [Exceptions]

BaseRank ChunkID  Section     BiScore  CE Score  ReRank 
1        122      Article 50  0.430    -0.907    1      
2        21       Article 5   0.424    -1.951    2      
3        267      ANNEX III   0.317    -7.000    4      
4        19       Article 5   0.308    -8.372    5      
5        11       Article 3   0.307    -3.200    3      
6        27       Article 5   0.292    -10.340   6      
7        4        Article 2   0.285    -11.075   9      
8        3        Article 2   0.271    -11.221   10     
9        26       Article 5   0.269    -10.888   8      
10       25       Article 5   0.258    -10.559   7      

--- Answer 1 (baseline top 3) ---
 Yes, the use of AI systems to infer emotions of a natural person in the workplace is prohibited under Article 5.

There is one exception: the prohibition does not apply if the AI system is intended to be put in place or into the market for medical

### 6c. Optional: discover your own failures automatically
Paste any questions here (no gold pattern needed). The table shows how much the top 3 changed, which points
you to queries worth reading closely.

In [39]:
EXTRA_QUERIES = [
    "What must providers of general-purpose AI models with systemic risk do?",
    "Are AI systems used only for military purposes covered by the Act?",
    "What are the transparency obligations for deepfakes?",
    "Which high-risk AI systems in employment are listed?",
]
rows = []
for q in EXTRA_QUERIES:
    r = run_pipelines(q, with_llm=False)
    moved_in = [c for c in r["rerank_top3"] if c not in r["baseline_top3"]]
    rows.append({"query": q[:70], "chunks promoted into top3": moved_in,
                 "their baseline ranks": [r["baseline_rank"].index(c) + 1 for c in moved_in]})
pd.DataFrame(rows)

,query,chunks promoted into top3,their baseline ranks
0,What must providers of general-purpose AI mode...,[127],[5]
1,Are AI systems used only for military purposes...,"[2, 272]","[9, 6]"
2,What are the transparency obligations for deep...,"[121, 131]","[4, 5]"
3,Which high-risk AI systems in employment are l...,"[268, 28]","[8, 9]"


## 7. Latency profiling
Each stage is repeated `N_RUNS` times (after one warm-up) and averaged. LLM time is reported separately because it
is the same size for both pipelines and dominates end-to-end time.

In [40]:
N_RUNS = 10
bench_queries = [q for _, q, _ in TEST_QUERIES]
_ = run_pipelines(bench_queries[0], with_llm=False)   # warm-up

t_ret, t_rer = [], []
for _ in range(N_RUNS):
    for q in bench_queries:
        r = run_pipelines(q, with_llm=False)
        t_ret.append(r["t_retrieve"]); t_rer.append(r["t_rerank"])

llm_times = [(r["t_llm1"], r["t_llm2"]) for r in all_runs]

lat = pd.DataFrame({
    "Stage": ["Pipeline 1: bi-encoder retrieval (embed query + FAISS top-10)",
              "Pipeline 2 extra: cross-encoder re-rank of 10 pairs",
              "Pipeline 1 total retrieval",
              "Pipeline 2 total retrieval",
              "LLM call (pipeline 1, avg)",
              "LLM call (pipeline 2, avg)"],
    "Avg ms": [np.mean(t_ret) * 1000,
               np.mean(t_rer) * 1000,
               np.mean(t_ret) * 1000,
               (np.mean(t_ret) + np.mean(t_rer)) * 1000,
               np.mean([a for a, _ in llm_times]) * 1000,
               np.mean([b for _, b in llm_times]) * 1000],
}).round(1)
lat

,Stage,Avg ms
0,Pipeline 1: bi-encoder retrieval (embed query ...,6.4
1,Pipeline 2 extra: cross-encoder re-rank of 10 ...,42.5
2,Pipeline 1 total retrieval,6.4
3,Pipeline 2 total retrieval,48.9
4,"LLM call (pipeline 1, avg)",2251.0
5,"LLM call (pipeline 2, avg)",3652.5
